# Workflow 1: Calibration, Segmentation, Pre-processing, Model Deployment and, Results Export

## Inputs

In [1]:
#Import libraries
import os
import hyp_camera as hyp
import traceback
import logging
import sys
import pandas as pd
import glob

In [ ]:
# === Initialize Project Paths and Folders ===

# Define the project name and base working directory
project_name = "PRUEBAS_fig7"
working_directory = "C:/Users/Pheno/Documents/database_almondcv3/"

# Define the project directory and log file path
project_directory = os.path.join(working_directory, project_name)
log_file = os.path.join(project_directory, "all_logs.txt")

# Define paths for session data, model, and output results
# Structure of the session file is provided in a info.txt file in the repository
path_sessions = os.path.join(working_directory, "experimental_sessions")
results_directory = os.path.join(project_directory, f"Results_{project_name}")

# Segmented pics are saved here
segmented_pseudorgb_directory = os.path.join(results_directory, "PseudoRgb_Segmented")

# Create the necessary directories if they don't already exist
if not os.path.exists(project_directory):
    os.makedirs(project_directory)  # Create project directory

if not os.path.exists(results_directory):
    os.makedirs(results_directory)  # Create results output directory

if not os.path.exists(segmented_pseudorgb_directory):
    os.makedirs(segmented_pseudorgb_directory)  # Create subfolder for segmented pseudo-RGB images


In [ ]:
# === Select Calibration Type ===
#
# Calibration is applied **pixelwise for each line of the hyperspectral image**.
# This means that for each pixel column (or line) across the image, the reflectance is calculated individually
# based on the sensor response and known white reference values.
#
# Two calibration types are supported:
#
# 1. "linear_2" – Linear calibration using two points:
#    - 0% reflectance (dark reference)
#    - X% reflectance (e.g., 50%)
#    A straight-line model is fitted for each pixel column.
#
# 2. "quadratic_3" – Quadratic calibration using three points:
#    - 0% reflectance (dark reference)
#    - X% reflectance (e.g., 50%)
#    - Y% reflectance (e.g., 90%)
#    A second-degree polynomial is fitted, allowing correction for sensor non-linearity.
#
# The variable `pretrained` indicates whether to load a pre-calculated calibration model (.pkl format).
# If `pretrained = True`, the model is expected to be already stored in the session directory in .pkl format.
# This file must have been generated using the `load_calibration_model` method 
# in the `hyp_session` class previously.
#
# If `pretrained = False`, the calibration will be computed from the provided reference files (`theor2_path`, `theor3_path`)
# and a new model will be saved automatically for future use.

# --- Example for Quadratic Calibration ---
calibration_type = "quadratic_3"
pretrained = True
theor2_path = r"C:\Users\Pheno\Desktop\Almond_CV\Valores_reflectancia_blancos\50%_22042119.csv"
theor3_path = r"C:\Users\Pheno\Desktop\Almond_CV\Valores_reflectancia_blancos\90%_23081005.csv"

# # --- Example for Linear Calibration ---
# calibration_type = "linear_2"
# pretrained = False

# Reference Reflectance File Structure
#
# The file contains spectral reference data, typically for a standard white or reference panel.
# It has two columns, separated by commas:
#
# 1. nm  : Wavelength in nanometers (numeric)
# 2. R   : Reflectance value at that wavelength (numeric, usually 0-100)
#
# Example:
# nm,R
# 250,51.540694
# 251,51.505076
# theor2_path = r"C:\Users\Pheno\Escritorio\Almond_CV\almond_cv3_processing\Valores_reflectancia_blancos\50%_22042119.csv"



In [ ]:
# === Select segmentation type ===

# Option 1: AI YOLO fine-tuned model, according to Mas-Gómez 2023 et al.
segment_type="model_ai"
model_path = os.path.join(working_directory, "YOLO_models/hyp_1303_band_70_320_improved.pt")
kwargs = {"model_path": model_path, "working_directory": results_directory, "img_size":320, "conf":0.85}
masks_path=None

# Option 2: Use previously prepared masks.
# Masks must have the same name as the corresponding image file and be in PNG format.

# segment_type="manual_masks"
# masks_path = os.path.join(working_directory, "masks_rgb")
# kwargs = {}

In [ ]:
# === Select preprocessing type ===
# 
# The list below defines the spectral preprocessing types to be applied. 
# "RAW" refers to the unprocessed reflectance data. "SNV" (Standard Normal Variate) and "MSC" (Multiplicative Scatter Correction) are commonly used normalization techniques to reduce scatter effects.
# 
# "SG1_Wx_P2" indicates the application of the Savitzky-Golay filter (first derivative, denoted by SG1), 
# where "W" stands for the window size (e.g., W5 means a window size of 5) and "P2" indicates the use of a 2nd-order polynomial.
# 
# Combinations such as "SNV_SG1_W5_P2" or "SG1_SNV_W5_P2" indicate whether SNV is applied before or after the Savitzky-Golay filter.
# All combinations listed below are pre-defined and can be used to evaluate which preprocessing method performs best for your analysis.

# preproc=["RAW","SNV", "MSC", "SG1_W5_P2", "SNV_SG1_W5_P2","SG1_SNV_W5_P2",
#                                                                        "SG1_W10_P2", "SNV_SG1_W10_P2", "SG1_SNV_W10_P2",
#                                                                         "SG1_W15_P2", "SNV_SG1_W15_P2", "SG1_SNV_W15_P2",
#                                                                          "SG1_W20_P2", "SNV_SG1_W20_P2", "SG1_SNV_W20_P2",
#                                                                           "SG1_W25_P2", "SNV_SG1_W25_P2", "SG1_SNV_W25_P2"
#                                                                             ]

preproc=["SG1_W15_P2"]

#If you want to cut head or tail in spectra
bands=[900,1750,2]
cut_head=25
cut_tail=-40

In [ ]:
# === Result Export Parameters ===
#
# `median_cal = True` enables the calculation of the **median** in addition to the mean when exporting results.
# This can be useful for capturing the central tendency more robustly, especially in the presence of outliers.
# ⚠️ Note: Computing the median is **slower** than just calculating the mean, especially for large datasets.
#
# `save = False` controls whether the processed images are saved.
# If set to `True`, the images will be saved in a **compressed format** depending on the last processing step applied:
# - If segmentation was used, segmented images will be saved.
# - If preprocessing (e.g., SNV, SG) was applied, the preprocessed images will be saved.
# - If calibration was applied, the calibrated reflectance images will be saved.
#
# This flag allows you to avoid unnecessary I/O operations when only analysis results are needed.

median_cal = True
save = False

In [ ]:
# === File with PLS info for predicting ===

# Trait            : str
#     The name of the trait to predict (e.g., 'Fiber', 'Protein').

# Metric           : str
#     The preprocessing or spectral metric used (e.g., 'Mean_SG1_W15_P2').

# model_path       : str
#     Full path to the saved PLS or iPLS model (.pkl).

# scale_path       : str or None
#     Optional path to a saved scaler to apply to the spectral data before prediction.

# Bands_selected   : list or str (optional, only for iPLS)
#     List of band indices to use for prediction (relative to the spectral columns in df_long). 
#     Can be a string that will be parsed as a list, e.g., '[0, 2, 5, 10]'
# heatmap_scale     : str (optional)
#     String defining the scaling range for the predicted heatmap values, usually in the format 'min_max' (e.g., '50_190').
#     The predicted pixel values will be rescaled to the 0–255 range based on these min and max values before visualization.
#     If not provided, a default scaling may be applied, which could affect the relative color intensity in the heatmap.
# IMPORTANT:
# The spectral columns in X must follow the exact order that was used when training the model.
# - For iPLS models: only the selected bands (Bands_selected), **in the same order as in the original input**.
# Any change in order will lead to incorrect predictions, because the model coefficients are aligned with that order.

model_pls_df=r"C:\Users\Pheno\Documents\database_almondcv3\RESULTADOS_PAPER\MODELOS_SELECCIONADOS_PAPER\selected_models.txt"

# Pixel averaging for smoothing picture results

pic_smoothing_px=8

In [ ]:
# Activating save logs disables console logging; logs appear only in this cell and in txt.
# === Logger Setup with stdout/stderr redirection ===

# import logging
# import sys
# import os

# # Ruta del log
# log_file = os.path.join(project_directory, "all_logs.txt")

# # Clase para redirigir stdout/stderr al logger
# class StreamToLogger:
#     def __init__(self, logger, level):
#         self.logger = logger
#         self.level = level

#     def write(self, message):
#         if message.strip():  # evitar líneas vacías
#             self.logger.log(self.level, message.strip())

#     def flush(self):
#         pass

# # Limpiar handlers antiguos (importante en notebooks)
# for handler in logging.root.handlers[:]:
#     logging.root.removeHandler(handler)

# # Configurar logger
# logging.basicConfig(
#     level=logging.DEBUG,
#     format="%(asctime)s - %(levelname)s - %(message)s",
#     handlers=[
#         logging.FileHandler(log_file, mode='w', encoding='utf-8'),
#         logging.StreamHandler(sys.stdout)
#     ]
# )

# # Obtener logger y redirigir stdout/stderr
# logger = logging.getLogger()
# sys.stdout = StreamToLogger(logger, logging.INFO)
# sys.stderr = StreamToLogger(logger, logging.ERROR)

# logger.info("Logging configured successfully.")


In [ ]:
#Logs in console

import logging
import os
import sys

log_file = os.path.join(project_directory, "all_logs.txt")

# Limpiar handlers antiguos
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file, mode='w', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger()
logger.info("Logging configured successfully.")

print("This print goes ONLY to the console.")
logger.info("This log goes to BOTH console and file.")

## Select process

In [ ]:
# Configuration flags
calibrate = True
segment = True
preprocessing = True
predict= True
obtain_pic_predicted= True

# Determine process and subdir
if calibrate and segment and preprocessing:
    process_name = "calibrate + segment + preprocessing"
    subdir = "HYP/RAW/"
elif calibrate and segment:
    process_name = "calibrate + segment"
    subdir = "HYP/RAW/"
elif calibrate and preprocessing:
    print("No process possible, activate segment. Exiting.")
elif segment and preprocessing:
    process_name = "segment + preprocessing"
    subdir = "HYP/CAL/"
elif segment:
    process_name = "segment"
    subdir = "HYP/CAL/"
elif preprocessing:
    process_name = "preprocessing"
    subdir = "HYP/CAL_SEG_BATCH/"
elif calibrate:
    process_name = "calibrate"
    subdir = "HYP/RAW/"
else:
    process_name = "none"
    subdir = None

print(f"\nActivated process: {process_name}")

# If no process is activated, exit
if subdir is None:
    print("No process activated. Exiting.")
else:
    # Check existence of subdir in each session folder
    print("\nSession status:")
    for session_name in os.listdir(path_sessions):
        session_path = os.path.join(path_sessions, session_name)
        if os.path.isdir(session_path):
            subdir_path = os.path.join(session_path, subdir)
            if os.path.exists(subdir_path):
                print(f"[{session_name}] {subdir} is ready ✅")
            else:
                print(f"[{session_name}] {subdir} is missing ❌")


#### Execute

In [ ]:
# === Loop over sessions and log all actions ===

# Get list of sessions
sessions = os.listdir(path_sessions)

for session in sessions:
    try:
        logger.info(f"Processing session: {session}")

        # Initialize session object
        object_session = hyp.hyp_session(
            session=session,
            path_sessions=path_sessions,
            results_directory=results_directory
        )

        # Load results table
        object_session.load_table(filename="result_table_general.txt")

        # Load calibration model
        object_session.load_calibration_model(
            pretrained=pretrained,
            theor50_path=theor2_path,
            theor90_path=theor3_path,
            type=calibration_type
        )

        # Run main pipeline
        object_session.calibrate_segment_preprocess(
            masks_path=masks_path,
            segmented_pseudorgb_directory=segmented_pseudorgb_directory,
            median_cal=median_cal,
            calibrate=calibrate,
            subdir=subdir,
            segment=segment,
            preprocessing=preprocessing,
            save=save,
            preproc=preproc,
            segment_type=segment_type,
            cut_head=cut_head,
            cut_tail=cut_tail,
            predict=predict,
            model_pls_df=model_pls_df,
            bands=bands,
            pic_smoothing_px=pic_smoothing_px,
            obtain_pic_predicted=obtain_pic_predicted,
            **kwargs
        )

    except Exception as e:
        logger.error("An error occurred during session processing:")
        logger.error(f"Error type: {type(e).__name__}")
        logger.error(f"Error message: {str(e)}")
        logger.error("Detailed traceback:")
        logger.error(traceback.format_exc())
        continue


## Join sessions results

In [ ]:
# Search for all files that start with "all_results" in the folder
files = glob.glob(f"{results_directory}/all_results*.txt")

# Read and concatenate the files
dfs = [pd.read_csv(file, sep="\t") for file in files]
concatenated_df = pd.concat(dfs, ignore_index=True)

# Export the DataFrame to a new TXT file
concatenated_df.to_csv(f"{results_directory}/all_results_joined.txt", sep="\t", index=False)


In [ ]:
import pandas as pd
import numpy as np
import joblib

# -----------------------------
# Paths
# -----------------------------
data_path = r"C:\Users\Pheno\Documents\database_almondcv3\pruebaaaaaaa\Results_pruebaaaaaaa\all_results_0409_3.txt"   # tu txt
model_path = r"C:\Users\Pheno\Documents\database_almondcv3\RESULTADOS_PAPER\MODELOS_SELECCIONADOS_PAPER\Models_kernel\PLS_Fats_Mean_SG1_W15_P2_11comp.pkl"

# -----------------------------
# Load model
# -----------------------------
model = joblib.load(model_path)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(data_path, sep="\t")

# columna espectral
spectral_col = "Mean_SG1_W15_P2"

# -----------------------------
# Predict per sample
# -----------------------------
results = []

for sample_id, g in df.groupby("ID"):

    # ordenar por banda
    g = g.sort_values("Band")

    # extraer espectro
    spectrum = g[spectral_col].values

    if len(spectrum) != 360:
        print(f"Warning: {sample_id} tiene {len(spectrum)} bandas")
        continue

    # reshape a (1,360)
    spectrum = spectrum.reshape(1, -1)

    # predicción
    predicted_value = model.predict(spectrum)[0]

    print(f"ID: {sample_id} -> {predicted_value}")

    results.append({
        "ID": sample_id,
        "Prediction": predicted_value
    })

# -----------------------------
# Save results
# -----------------------------
results_df = pd.DataFrame(results)
print(results_df)